# Preprocessing

In [1]:
import re 
import time
import os
import random
from concurrent.futures import ThreadPoolExecutor


import pandas as pd
import xarray as xr
from xarray.backends.api import open_datatree
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np
import pyproj
import boto3
import earthaccess

from dotenv import load_dotenv

load_dotenv()

auth = earthaccess.login()

In [2]:
tspan = ('2025-09-02T00:00:00Z', '2025-09-04T00:00:00Z')

results = earthaccess.search_data(
    short_name = "PACE_OCI_L1B_SCI",
    temporal=tspan,
    count=10000
)

print("total results: ", len(results))
[list(res.values())[1]['SpatialExtent']['HorizontalSpatialDomain']['Geometry']['GPolygons'][0]['Boundary']['Points'][0] for res in results]

total results:  289


[{'Latitude': -57.22143, 'Longitude': -137.88263},
 {'Latitude': -41.63528, 'Longitude': -153.30138},
 {'Latitude': -24.57651, 'Longitude': -161.69112},
 {'Latitude': -6.96488, 'Longitude': -167.0742},
 {'Latitude': 16.16699, 'Longitude': -171.74113},
 {'Latitude': 34.05639, 'Longitude': -174.11717},
 {'Latitude': 51.86375, 'Longitude': -175.11531},
 {'Latitude': 69.44675, 'Longitude': -172.41042},
 {'Latitude': 80.03294, 'Longitude': -160.91325},
 {'Latitude': -57.25105, 'Longitude': -162.4222},
 {'Latitude': -41.6603, 'Longitude': -177.87134},
 {'Latitude': -24.61347, 'Longitude': 173.73668},
 {'Latitude': -7.00259, 'Longitude': 168.34932},
 {'Latitude': 16.13982, 'Longitude': 163.67842},
 {'Latitude': 34.01849, 'Longitude': 161.30258},
 {'Latitude': 51.82583, 'Longitude': 160.30176},
 {'Latitude': 69.41872, 'Longitude': 162.98076},
 {'Latitude': 85.13401, 'Longitude': -151.54173},
 {'Latitude': 76.43255, 'Longitude': -51.56736},
 {'Latitude': -57.27102, 'Longitude': 173.02684},
 {'L

In [3]:
[res.dataviz_links() for res in results]

[['https://oceandata.sci.gsfc.nasa.gov/browse_images/PACE_OCI.20250902T004050.L1B.V3.nc.png?file_path=PACE_OCI/IMAGES/EDBRS/2025/0902'],
 ['https://oceandata.sci.gsfc.nasa.gov/browse_images/PACE_OCI.20250902T004550.L1B.V3.nc.png?file_path=PACE_OCI/IMAGES/EDBRS/2025/0902'],
 ['https://oceandata.sci.gsfc.nasa.gov/browse_images/PACE_OCI.20250902T005050.L1B.V3.nc.png?file_path=PACE_OCI/IMAGES/EDBRS/2025/0902'],
 ['https://oceandata.sci.gsfc.nasa.gov/browse_images/PACE_OCI.20250902T005550.L1B.V3.nc.png?file_path=PACE_OCI/IMAGES/EDBRS/2025/0902'],
 ['https://oceandata.sci.gsfc.nasa.gov/browse_images/PACE_OCI.20250902T010050.L1B.V3.nc.png?file_path=PACE_OCI/IMAGES/EDBRS/2025/0902'],
 ['https://oceandata.sci.gsfc.nasa.gov/browse_images/PACE_OCI.20250902T010550.L1B.V3.nc.png?file_path=PACE_OCI/IMAGES/EDBRS/2025/0902'],
 ['https://oceandata.sci.gsfc.nasa.gov/browse_images/PACE_OCI.20250902T011050.L1B.V3.nc.png?file_path=PACE_OCI/IMAGES/EDBRS/2025/0902'],
 ['https://oceandata.sci.gsfc.nasa.gov/br

# Iteration 1: Save Entire Granule

In [ ]:
granules = ['20250903T144236', '20250903T144736','20250903T145236','20250903T145736', '20250903T150236']

scan_ids = range(1065)

def store_granule(granule_id, scan_ids=scan_ids):
  print("++++++++++++")
  print(f"Processing granule {granule_id}")
  results = earthaccess.search_data(
    short_name="PACE_OCI_L1B_SCI",
    granule_name=f"PACE_OCI.{granule_id}.L1B.V3.nc",)

  paths = earthaccess.open(results)
  datatree = open_datatree(paths[0])
  dataset = xr.merge(datatree.to_dict().values())

  for scan_id in scan_ids:
    get_tostore_vector(dataset, scan_id, granule_id)
    if not scan_id % 50:
      print(f"scan_id: {scan_id} completed for granule {granule_id}")


def get_tostore_vector(dataset, scan_id, granule_id):
  rhot_blue_arr = dataset["rhot_blue"].isel(scans=scan_id).values
  rhot_red_arr = dataset["rhot_red"].isel(scans=scan_id).values
  rhot_SWIR_arr = dataset["rhot_SWIR"].isel(scans=scan_id).values

  tostore = np.vstack([rhot_blue_arr, rhot_red_arr, rhot_SWIR_arr]).T

  path = f"s3://compressive-sensing/{granule_id}/scan{scan_id}.parquet"

  df = pd.DataFrame(tostore)
  df.to_parquet(
      path,
      index=False,
      engine="pyarrow",
  )


# 2. Sampling - 1% of Granules

In [ ]:

granules = [re.search(r"(\d{8}T\d{6})", res.dataviz_links()[0]).group(1) for res in results]

data = []

def process_granule(granule_id):
    results = earthaccess.search_data(
    short_name="PACE_OCI_L1B_SCI",
    granule_name=f"PACE_OCI.{granule_id}.L1B.V3.nc",)

    random_scan = random.randint(0, 1065)

    print(f"Processing granule {granule_id} with random scan {random_scan}")

    paths = earthaccess.open(results)
    datatree = open_datatree(paths[0])
    dataset = xr.merge(datatree.to_dict().values())

    rhot_blue_arr = dataset["rhot_blue"].isel(scans=random_scan).values
    rhot_red_arr = dataset["rhot_red"].isel(scans=random_scan).values
    rhot_SWIR_arr = dataset["rhot_SWIR"].isel(scans=random_scan).values
    tostore = np.vstack([rhot_blue_arr, rhot_red_arr, rhot_SWIR_arr]).T

    path = f"s3://compressive-sensing/samples/{granule_id}/scan{random_scan}.parquet"

    df = pd.DataFrame(tostore)
    df.to_parquet(
        path,
        index=False,
        engine="pyarrow",
    )

for i, granule_id in enumerate(granules[:3]):
    process_granule(granule_id)
    print(f"{i} granules processed")

Processing granule 20250902T004050 with random scan 94


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

ImportError: Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.